# CCD 관측 자료 전처리

* 이 노트북을 구글 코랩에서 실행하고자 한다면 [파일] - [드라이브에 사본 저장]을 하여 본인의 소유로 만든 후에 코드를 실행하거나 수정할 수 있습니다.

* 이 파일은 실제 수업에 사용하므로 필요에 따라 예고 없이 변경될 수 있습니다.

* If you have any questions or comments on this document, please email me(Kiehyun.Park@gmail.com).

* 이 파일(문서)는 공교육 현장에서 수업시간에 자유롭게 사용할 수 있으나, 다른 목적으로 사용할 시에는 사전에 연락을 주셔서 상의해 주시기 바랍니다.



천체 관측 중 CCD(charge couple device) 관측 자료를 이용하여 측광의 기본인 전처리 과정을 다룹니다.

## 필요한 환경

이 프로젝트를 위해서는 아래의 모듈이 필요합니다.

> numpy, matplotlib, ccdproc, astropy, gdown, ysfitsutilpy, ysphotutilpy, ysvisutilpy, version_information



### 구글 코랩에 한글 폰트 설치

matplotlib에서 한글을 사용하기 위해서는 한글 폰트가 필요하다. 구글 코랩에서 현재의 Jupyter notebook을 실행한다면 아래 코드를 실행 한 후 런타임 다시 시작을 해 줘야 한글을 사용할 수 있을 것이다.

In [ ]:
# !sudo apt-get install -y fonts-nanum
# !sudo fc-cache -fv
# !rm ~/.cache/matplotlib -rf

### 런타임 다시 시작

위의 셀을 실행한 다음 반드시 다음 과정을 잊지 말자.

* [메뉴]-[런타임]-[런터임 다시 시작]

* [메뉴]-[런타임]-[이전 셀 실행]을 해주어야 한다.

### 한글 폰트 사용

위에서 한글 폰트를 설치하고, 런타임 다시시작을 했다면 구글 코랩에서 폰트 경로를 설정하여 한글 사용이 가능해 진다.

In [ ]:
#visualization
import matplotlib as mpl
import matplotlib.pylab as plb
import matplotlib.pyplot as plt

# 브라우저에서 바로 그려지도록
%matplotlib inline

# 그래프에 retina display 적용
%config InlineBackend.figure_format = 'retina'

# Colab 의 한글 폰트 설정
plt.rc('font', family='NanumBarunGothic')

# 유니코드에서  음수 부호설정
mpl.rc('axes', unicode_minus=False)

### 모듈 설치 및 버전 확인

아래 셀을 실행하면 이 노트북을 실행하는데 필요한 모듈을 설치하고 파이썬 및 관련 모듈의 버전을 확인할 수 있습니다.

In [ ]:
import importlib, sys, subprocess
packages = "numpy, matplotlib, ccdproc, astropy, gdown, ysfitsutilpy, ysphotutilpy, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"**** {pkg} module is now being installed.")
    else:
        print(f"******** {pkg} module is already installed.")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

## 데이터 저장

### 데이터 저장 폴더 생성

데이터를 저장할 폴더를 "20170220_m35" 라는 이름으로 생성해 보겠습니다.

* 만약 리눅스 시스템 이라면 shell 명령어로 가능한데, "!"를 붙이면 shell 명령어를 실행할 수 있습니다.
> !mkdir 20170220_m35

아래 코드를 실행하면 OS의 영향을 받지 않기 위하여 pathlib을 사용하여 폴더를 생성할 수 있습니다.

In [ ]:
import os
from pathlib import Path
BASEPATH = Path("./")
save_dir_name = "20170220_m35"
print(f"BASEPATH: {BASEPATH}")

if not (BASEPATH/save_dir_name).exists():
    os.mkdir(str(BASEPATH/save_dir_name))
    print (f"{str(BASEPATH/save_dir_name)} is created...")
else :
    print (f"{str(BASEPATH/save_dir_name)} is already exist...")

shell 명령어로 폴더가 생성되었는지 확인해 봅니다.

폴더 생성을 확인하는 또 다른 방법은 이 창 오른쪽에서 [파일] 목록을 확인해 볼 수 있습니다.

In [ ]:
# !ls -al
!dir 

### CCD 관측 데이터 다운로드

경기과학고등학교 천문대에서 CCD로 관측한 이미지 (예제 fits files)을 구글 드라이브에 저장해 놓았습니다. wget 명령을 사용하여 다음과 같이 구글 코랩 [작업 영역]에 다운로드 받아 보겠습니다.

*   bias frame : 5장
*   dark frame : 5장
*   flat frame : 5장
*   object frame : 10장

파일의 갯수가 많아서 하나의 압축파일로 저장해 놓은 것을 받아 보도록 하겠습니다.



In [ ]:
# #wget을 이용하여 데이터 다운로드
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=19m_ypEDifTNS5rtDBs_XljlLKz55dRgl' -O ./20170220_m35/bias-e-2X2-001.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1gt5pJpZ0NzHOb9n9RfFlgdQjdeanhnTS' -O ./20170220_m35/bias-e-2X2-002.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1IZCgDETUnTz98z_utx3bvyDtJdAkHnHF' -O ./20170220_m35/bias-e-2X2-003.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1BZxecgGBZzgrFpvpCL0IJxOOVj-jjFcG' -O ./20170220_m35/bias-e-2X2-004.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1O_OB5Bt71eCF0Ao4smMYDGEN8BmRKye6' -O ./20170220_m35/bias-e-2X2-005.fits

# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1I2uM-IwP7KV9nbJ-9geHw4eJPJQEnkga' -O ./20170220_m35/dark-e-2X2-011.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1HsNv6xWgU3LyI5moL1S8xNLDJRGqYuAF' -O ./20170220_m35/dark-e-2X2-012.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1Aa4OjTJDDFiBiq3QiKn4d8Wk0ywzUKxq' -O ./20170220_m35/dark-e-2X2-013.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=147xozc1vr3qoxX0XgiUCpF68_e0oQtNB' -O ./20170220_m35/dark-e-2X2-014.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1vxfUgrHBRSCIT3nZ1UcRhPn3zQYw0OQ6' -O ./20170220_m35/dark-e-2X2-015.fits

# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1kLAyz5_PKj5xSMNMI8SaMqYCJJscO20y' -O ./20170220_m35/flatV-e-2X2-021.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1kLAyz5_PKj5xSMNMI8SaMqYCJJscO20y' -O ./20170220_m35/flatV-e-2X2-022.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1L9NkrV76Jzv7_lGOXcxMnpk48LrvERf0' -O ./20170220_m35/flatV-e-2X2-023.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1u4n1ElNSmvmLPj8mphpNViC18QXhgwEp' -O ./20170220_m35/flatV-e-2X2-024.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1vG3q4rfsQCSbrYAzYd3eywIzINRF_8Jr' -O ./20170220_m35/flatV-e-2X2-025.fits

# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1NJAWZ-g3Bsdo1MFmaxGApkGplIRCWL_M' -O ./20170220_m35/M35-g3035781.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1IVSURFu9h1jqcANWYAzWNsDDEirmLNBl' -O ./20170220_m35/M35-g3035782.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1hBenRbavBdmqPhpy7rJAFy6t0H62qoeP' -O ./20170220_m35/M35-g3035783.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1QjX5xLaS_ShluL8gGuRLKXcStLOkv6DP' -O ./20170220_m35/M35-g3035784.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1SPzi05NfMZSuEyBU_GO8gwUkQ2heMqw9' -O ./20170220_m35/M35-g3035785.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1GgWOwmSk0LJkJlDQc5SnJtHVWqbLSsrj' -O ./20170220_m35/M35-g3035786.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1iLmCRpluVf5da9xeQMjEhou2A_xl4CI9' -O ./20170220_m35/M35-g3035787.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1bQCoYtzXCwGJ10ZkL61P7l7KQgronoHg' -O ./20170220_m35/M35-g3035788.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1Okq7iPBIkckzPepPuR9Q1Z10tbzN1kZ4' -O ./20170220_m35/M35-g3035789.fits
# !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1aV0vBvlZi7w181wLULRaSYuMaJN-zZfe' -O ./20170220_m35/M35-g3035790.fits


### gdown

구글 드라이브에 저장된 파일을 wget으로 다운받는 경우 용량의 제한이 있습니다. 이를 해결할 수 있는 방법 중의 하나가 gdown을 이용하는 것이다.

In [ ]:
fname = "20170220_m35.zip"
fid = "1o2YV3Z_G5BIl-MtKg9ng2lTV6W3eHMmU"

# gdown을 이용(나의 구굴드라이브에서 공유한 파일을 다운로드)
!gdown {fid} -O {save_dir_name}/{fname}

In [ ]:
# 분리된 압축파일 다운로드
# fname_bias = "bias_frames.zip"
# fid_bias = "1JKDuAHfkdNKqVOp9Jkb6N_0j0FkvNwb8"
# fname_dark = "dark_frames.zip"
# fid_dark = "1Nne7jaGTUTnRb3XTHSAh7FRrjzq6MHaR"
# fname_flat = "flat_frames.zip"
# fid_flat = "1AP_0Fpdyq-Fy9hC6YZbENIURM5BNgSRy"
# fname_light = "light_frames.zip"
# fid_light = "1OvV7fcwQ550H4iCqRZNpvFYOhK-5RDEs"

# # wget을 이용(나의 구굴드라이브에서 공유한 파일을 구글 코랩에서 사용할 경우)
# !wget --no-check-certificate "https://docs.google.com/uc?export=download&id={fid_bias}" -O {save_dir_name}/{fname_bias}
# !wget --no-check-certificate "https://docs.google.com/uc?export=download&id={fid_dark}" -O {save_dir_name}/{fname_dark}
# !wget --no-check-certificate "https://docs.google.com/uc?export=download&id={fid_flat}" -O {save_dir_name}/{fname_flat}
# !wget --no-check-certificate "https://docs.google.com/uc?export=download&id={fid_light}" -O {save_dir_name}/{fname_light}

# # gdown을 이용(나의 구굴드라이브에서 공유한 파일을 다운로드)
# #!gdown {fid} -O {save_dir_name}/{fname}

### 데이터 확인

* 코랩을 사용할 경우에는 오른쪽의 [파일]창에서 확인할 수 있습니다.
* linux shell 명령어로 다음과 같이 확인해 볼 수 있습니다.
> ls -l 20170220_m35

OS의 영향을 받지 않고 파이썬으로 확인하는 방법은 아래와 같이 하면 됩니다.

In [ ]:
from pathlib import Path
fpaths = sorted(list((BASEPATH/save_dir_name).glob('*.zip')))
print(f"fpaths: {fpaths}")
print(f"len(fpaths): {len(fpaths)}")

### 압축 풀기


In [ ]:
import shutil

shutil.unpack_archive(str(fpaths[0]), str(BASEPATH/save_dir_name))
print(str(fpaths[0]), "is unpacked to", str(BASEPATH/save_dir_name))

In [ ]:
# from zipfile import ZipFile
# with ZipFile(str(fpaths[0]), 'r') as zip_ref:
#     zip_ref.extractall(str(BASEPATH/save_dir_name))
#     print(str(fpath[0]), "is unpacked to", str(BASEPATH/save_dir_name))

### 압축 해제된 파일 확인

shell 명령어로 파일이 생성되었는지 확인해 보자.
역시 마찬가지로 이 창 오른쪽에서 [파일] 목록을 확인해 볼 수 있습니다.

In [ ]:
fpaths = sorted(list((BASEPATH/save_dir_name).glob('*.fit*')))
print(f"fpaths: {fpaths}")
print(f"len(fpaths): {len(fpaths)}")

## About fits file

천문학에서는 확장자가 '.fits'인 파일이 널리 쓰입니다. FITS는 'Flexible Image Transport System'이란 뜻인데, 한 파일에 여러 프레임의 이미지를 저장할 수 있어서 관측 이미지를 담는 파일에 주로 쓰입니다.

Python에서는 주로 ccdproc, astropy 패키지를 이용하면 FITS 파일을 다룰 수 있게 됩니다.

### Load files

fits file을  읽어 확인해 보겠습니다.

In [ ]:
from astropy.io import fits

fpath = Path(fpaths[0])
print(f"fpath: {fpath}")
print(f"type(fpath): {type(fpath)}")

hdul = fits.open(str(fpath), unit="adu")

print("type(hdul) :", type(hdul))
print("type(hdul[0]) :", type(hdul[0]))

### header

ccds라는 이름에 HDUList들이 리스트 형태로 들어 있습니다. 각각의 hdulist는 2차원 이므로 index는 [0]번만 존재합니다.

In [ ]:
print("type(hdul[0].hedaer) :", type(hdul[0].header))
hdul[0].header

header는 key와 value가 들어 있습니다.

In [ ]:
print("hdul[0].hedaer['DATE-OBS'] :", hdul[0].header['DATE-OBS'])
print("type(hdul[0].hedaer['DATE-OBS']) :", type(hdul[0].header['DATE-OBS']))

### data

관측 자료는 numpy.ndarray 형태로 들어 있음을 알 수 있습니다.

In [ ]:
print("type(hdul[0].data) :", type(hdul[0].data))
print("hdul[0].data.dtype :", hdul[0].data.dtype)
print("hdul[0].data.shape :", hdul[0].data.shape)
print("hdul[0].data :", hdul[0].data)

## 바이어스 오차

바이어스 오차(bias error)는 CCD 관측 각 픽셀 초기 값이 0 이 되어야 하는데, 실제로는 그렇지 않기 때문에 생기는 noise 입니다. 이 noise는 픽셀들의 기계적인 원인 때문에 발생하는 것이며 관측시 각 픽셀들은 몇 개의 전자를 가지고 관측을 시작하게 됩니다.
이러한 전자에 의해 생기는 noise는 bias image를 촬영하여 다음의 과정으로 보정할 수 있습니다.

* 광자가 모이기 힘들 정도의 최대한 짧은 exposure time으로 bais image를 가능한 한 여려 장(10 ~ 20 frame) 촬영 합니다.

* all bias image를 median 방법으로 combine 하여 얻은 master bias image를 "bias-median.fits"로 저장 합니다.

* bias image는 filter에 무관합니다.

* 다만 저장시에 data type에 유의해야 하는데, ccdproc로 저장한후 data type을 확인하는 것이 좋습니다.


### bias 파일 리스트 만들기

파일 이름에 규칙성이 있다면 이를 이용하여 다음과 같이 파일 경로를 리스트로 만들어서 사용할 수 있습니다.

In [ ]:
from glob import glob
import os

bias_list = sorted(list((BASEPATH/save_dir_name).glob('*bias-e*.fit*')))
print(f"bias_list: {bias_list}")
print(f"len(bias_list): {len(bias_list)}")

In [ ]:
# from glob import glob
# import os

# bias_list = sorted(glob(os.path.join("./20170220_m35/", "bias*.fits")))
# print("bias_list : {}".format(bias_list))


In [ ]:
fpath = bias_list[0]
print(fpath)
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

from astropy.io import fits
hdul = fits.open(fpath)

# Create an ImageNormalize object
norm = simple_norm(hdul[0].data, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(hdul[0].data,
            origin='lower',
            #vmax = 3000,
            norm=norm,
                    )

im2 = axs[1].hist(hdul[0].data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'{fpath.name}', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {hdul[0].data.min()}, Mean value: {hdul[0].data.mean():.02f}, Max value: {hdul[0].data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

### master bias 만들기

ccdproc의 combime 함수를 이용하면 여러 장의 bias 이미지를 통계적으로 처리할 수 있습니다.

기본적으로는 여러장의 이미지로부터 각 픽셀의 통계값을 구하는데 기본은 average 값을 구하게 됩니다. 만약 median 값을 구하고자 하는 경우 method를 "median"을 입력해 주면 되며, 마스타 바이어스 프레임은 "median"이 더 유용할 것으로 생각됩니다.

In [ ]:
from ccdproc import combine

method = "median"

bias = combine(bias_list,       # ccdproc does not accept numpy.ndarray, but only python list.
               method = method,         # default is average so I specified median.
               unit='adu')

print("type(bias) :", type(bias))

### bias 확인하기

In [ ]:
print("dir(bias) :", dir(bias))
print("type(bias.data) :", type(bias.data))

### dtype 바꾸기



In [ ]:
print("bias.data.dtype :", type(bias.data.dtype))
bias.data = bias.data.astype(np.float32)
print("bias.data.dtype :", type(bias.data.dtype))

### 저장하기


In [ ]:
bias.write(f"{(BASEPATH/save_dir_name)}/master_bais_{method}.fits", overwrite =True)
print(f"{(BASEPATH/save_dir_name)}/master_bais_{method}.fits is created...")

### 화면에 출력하기

화면에 디스플레이 해 보자.

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(bias, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(bias.data,
            origin='lower',
            #vmax = 3000,
            norm=norm,
                    )

im2 = axs[1].hist(bias.data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'The master bias by median combine', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {bias.data.min()}, Mean value: {bias.data.mean():.02f}, Max value: {bias.data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

### 부분 확대해서 확인하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from astropy.nddata import Cutout2D

# Create an ImageNormalize object
norm = simple_norm(bias.data, 'sqrt')

fig_set = plt.figure(figsize=(16, 8))

ax1 = plt.subplot2grid((2,4), (0,0),
                    colspan=2, rowspan=2,
                    fig=fig_set)
im1 = ax1.imshow(bias.data,
            origin='lower',
            norm=norm,
            )
plt.colorbar(im1,
             ax = ax1,
             fraction=0.0455, pad=0.04)

#1/4 area
ax20 = plt.subplot2grid((2,4), (0,2),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = bias.data,
            position = (500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im20 = ax20.imshow(cut_hdu.data,
    origin='lower'
    )

ax20.set_ylabel('pixels')
ax20.grid(ls=':')
ax20.set_title(f'1/4 image area', fontsize=8)
plt.colorbar(im20,
            ax=ax20,
            fraction=0.0455, pad=0.04)

#2/4 area
ax21 = plt.subplot2grid((2,4), (0,3),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = bias.data,
            position = (bias.data.shape[1] - 500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im21 = ax21.imshow(cut_hdu.data,
    origin='lower'
    )

ax21.set_ylabel('pixels')
ax21.grid(ls=':')
ax21.set_title(f'2/4 image area', fontsize=8)
plt.colorbar(im21,
            ax=ax21,
            fraction=0.0455, pad=0.04)

#3/4 area
ax30 = plt.subplot2grid((2,4), (1,2),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = bias.data,
            position = (500, bias.data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im30 = ax30.imshow(cut_hdu.data,
    origin='lower'
    )

ax30.set_ylabel('pixels')
ax30.grid(ls=':')
ax30.set_title(f'3/4 image area', fontsize=8)
plt.colorbar(im30,
            ax=ax30,
            fraction=0.0455, pad=0.04)

#4/4 area
ax31 = plt.subplot2grid((2,4), (1,3),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = bias.data,
            position = (bias.data.shape[1] - 500, bias.data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im31 = ax31.imshow(cut_hdu.data,
    origin='lower'
    )

ax31.set_ylabel('pixels')
ax31.grid(ls=':')
ax31.set_title(f'4/4 image area', fontsize=8)
plt.colorbar(im31,
            ax=ax31,
            fraction=0.0455, pad=0.04)

plt.tight_layout()
plt.show()

### sigma_clip

1/4 영역은 매우 픽셀값이 큰 것을 확인할 수 있다. 오차가 큰 데이터는 지우고 다시 확인해 보자.


In [ ]:
from astropy.stats import sigma_clip

# (3) Sigma clip the bias
bias_clip = sigma_clip(bias)
# by default, astropy.stats.sigma_clip does "3-sigma, 10-iterations" clipping.
# You can tune the parameters by optional keywords (e.g., sigma, iters, ...).
# dark_clip is "numpy masked array". It contains data, mask, and filled_value.
# filled_value is the value which is used to fill the data which are masked (rejected).
# I will change filled_value to be the median value.

# (4) For rejected pixels, insert the median value
bias_clip.fill_value = np.median(bias_clip)
bias.data = bias_clip.filled() # ~.filled() gives the "data array using filled_value"

### 화면에 출력하기 (sigma_cliped bais)

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(bias, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(bias.data,
            origin='lower',
            #vmax = 3000,
            norm=norm,
                    )

im2 = axs[1].hist(bias.data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'The master bias by median combine', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {bias.data.min()}, Mean value: {bias.data.mean():.02f}, Max value: {bias.data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

## 열화 노이즈

암전류(dark current)는 CCD가 빛을 받지 않더라도 실리콘 격자의 열진동에 의하여 pixel에서 만들어진 신호를 말하며 이를 열화 노이즈라고 합니다. 즉, 실리콘 격자의 온도가 높으면 가전자의 일부가 튀어나오게 되는데 이러한 전자들이 신호를 만들게 됩니다. 이러한 암전류는 온도가 높을수록, 그리고 exposure time에 비례하게 나타납니다.

CCD를 냉각하여 dark current를 줄여줄 수 있으나, 완전히 제거할 수는 없으므로 dark frame을 촬영하여 이를 제거해 줍니다. dark current는 시간의 linear function 이므로 dark image의 exposure time($t$ second)를 $dark(t)$라 한다면

$ {\displaystyle{dark(1) = \frac{dark(t) - bias}{t}}}$ 이다.

대상 이미지의 노출 시간이 $t$인 경우 $dark(1)$에 $t$를 곱한 다음 'master dark'로 사용하면 dark frame을 촬영하는 시간을 절약할 수 있습니다. 다만 이 경우
dark current가 노출 시간에 정확하게 비례한다는 가정하에 수행하는 것입니다. 따라서 일반적으로 그러나 dark frmae은 대상을 촬영한 것과 exposeure time을 같게 하여 촬영하는 것이 일반적입니다.

이러한 noise를 dark current noise라고 하며 dark frame을 촬영하여 다음의 과정으로 보정이 가능하다.

* 셔터를 닫은 상태에서 calibration을 수행할 light frame과 같은 온도, 같은 exposure time으로 dark image를 가능한 한 여려 장(10 ~ 20 frame) 촬영한다.
* all dark image를 median 방법으로 combine 한 후 exposure time으로 나누어 "dark0-median.fits"로 저장한다.
* dark0 값에서 master bias를 빼준다.
* 그리고 3-sigma clip with 10 iterations 처리해 준다.
* set the rejected pixel values as the median value
* save it as "dark-median.fits"
* dark image는 filter에 무관하다.
* 다만 저장시에 data type에 유의해야 하는데, ccdproc로 저장한후 data type을 확인하는 것이 좋다.

### dark 파일 리스트 만들기

파일 이름에 규칙성이 있다면 이를 이용하여 다음과 같이 파일 경로를 리스트로 만들어서 사용할 수 있습니다.

In [ ]:
from glob import glob
import os

dark_list = sorted(list((BASEPATH/save_dir_name).glob('*dark-e*.fit*')))
print(f"dark_list: {dark_list}")
print(f"len(dark_list): {len(dark_list)}")

In [ ]:
# from glob import glob
# import os

# dark_list = sorted(glob(os.path.join("./20170220_m35/", "dark*.fits")))
# print("dark_list : {}".format(dark_list))

### master dark0 만들기

ccdproc의 combime 함수를 이용하면 여러 장의 dark 이미지를 통계적으로 처리할 수 있습니다.

기본적으로는 여러장의 이미지로부터 각 픽셀의 통계값을 구하는데 기본은 average 값을 구하게 됩니다. 만약 median 값을 구하고자 하는 경우 method를 "median"을 입력해 주면 되며, 마스타 바이어스 프레임은 "median"이 더 유용할 것으로 생각됩니다.

In [ ]:
import numpy as np
from ccdproc import combine, ccd_process
from astropy.stats import sigma_clip

method = "median"

dark0 = combine(dark_list,       # ccdproc does not accept numpy.ndarray, but only python list.
    method = method,         # default is average so I specified median.
    unit = 'adu')

# float32로 변환
dark0.data = dark0.data.astype(np.float32)

### 저장하기

In [ ]:
dark0.write(f"{(BASEPATH/save_dir_name)}/master_dark0_{method}.fits", overwrite =True)
dark0.write(f"{(BASEPATH/save_dir_name)}/master_dark0_{method}.fits", overwrite =True)

### 화면에 출력하기 (dark0)

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(dark0, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(dark0.data,
            origin='lower',
            #vmax = 3000,
            norm=norm,
                    )

im2 = axs[1].hist(dark0.data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'The master dark0 by median combine', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {dark0.data.min()}, Mean value: {dark0.data.mean():.02f}, Max value: {dark0.data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

### bias 빼주기

In [ ]:
# This dark isn't bias subtracted yet, so let's subtract bias:

# (1) Open master bias
#bias = CCDData.read(f"{(BASEPATH/save_dir_name)}/master_bais_{method}.fits")

# (2) Subtract bias:
dark = ccd_process(dark0, master_bias = bias)

dark.data = dark.data.astype(np.float32)
#print('dark', dark.data.min(), dark.data.max(), dark.data)
# This automatically does "dark-bias"
# You can do it by the function "subtract_bias", or just normal pythonic arithmetic.

### 화면에 출력하기

화면에 디스플레이 해 보자.

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(dark, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(dark.data,
            origin='lower',
            #vmax = 3000,
            norm=norm,
                    )

im2 = axs[1].hist(dark.data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'The master dark by median combine', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {dark.data.min()}, Mean value: {dark.data.mean():.02f}, Max value: {dark.data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

### (과제) sigma_clip

 bias와 같은 방법으로 dark를 sigma_cliped으로 일부 데이터 제거한 후 화면에 디스플레이 해 보자.

In [ ]:
#(과제) 이곳에 코딩을 완성하여 제출하시오.

from astropy.stats import sigma_clip

# (3) Sigma clip the dark

dark_clip = sigma_clip(dark)

# by default, astropy.stats.sigma_clip does "3-sigma, 10-iterations" clipping.
# You can tune the parameters by optional keywords (e.g., sigma, iters, ...).
# dark_clip is "numpy masked array". It contains data, mask, and filled_value.
# filled_value is the value which is used to fill the data which are masked (rejected).
# I will change filled_value to be the median value.

# (4) For rejected pixels, insert the median value

####
dark_clip.fill_value = np.median(dark.data)
dark.data = dark_clip.filled() # ~.filled() gives the "data array using filled_value"


### dark 저장하기


In [ ]:
# (5) Save
dark.write(f"{(BASEPATH/save_dir_name)}/master_dark_{method}.fits", overwrite =True)
dark.write(f"{(BASEPATH/save_dir_name)}/master_dark_{method}.fits", overwrite =True)

### 화면에 출력하기 (sigma_cliped dark)

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(dark, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(dark.data,
            origin='lower',
            #vmax = 3000,
            norm=norm,
                    )

im2 = axs[1].hist(dark.data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'The master dark by median combine', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {dark.data.min()}, Mean value: {dark.data.mean():.02f}, Max value: {dark.data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

### 부분 확대해서 확인하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from astropy.nddata import Cutout2D

# Create an ImageNormalize object
norm = simple_norm(dark.data, 'sqrt')

fig_set = plt.figure(figsize=(16, 8))

ax1 = plt.subplot2grid((2,4), (0,0),
                    colspan=2, rowspan=2,
                    fig=fig_set)
im1 = ax1.imshow(dark.data,
            origin='lower',
            norm=norm,
            )
plt.colorbar(im1,
             ax = ax1,
             fraction=0.0455, pad=0.04)

#1/4 area
ax20 = plt.subplot2grid((2,4), (0,2),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = dark.data,
            position = (500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im20 = ax20.imshow(cut_hdu.data,
    origin='lower'
    )

ax20.set_ylabel('pixels')
ax20.grid(ls=':')
ax20.set_title(f'1/4 image area', fontsize=8)
plt.colorbar(im20,
            ax=ax20,
            fraction=0.0455, pad=0.04)

#2/4 area
ax21 = plt.subplot2grid((2,4), (0,3),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = dark.data,
            position = (dark.data.shape[1] - 500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im21 = ax21.imshow(cut_hdu.data,
    origin='lower'
    )

ax21.set_ylabel('pixels')
ax21.grid(ls=':')
ax21.set_title(f'2/4 image area', fontsize=8)
plt.colorbar(im21,
            ax=ax21,
            fraction=0.0455, pad=0.04)

#3/4 area
ax30 = plt.subplot2grid((2,4), (1,2),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = dark.data,
            position = (500, dark.data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im30 = ax30.imshow(cut_hdu.data,
    origin='lower'
    )

ax30.set_ylabel('pixels')
ax30.grid(ls=':')
ax30.set_title(f'3/4 image area', fontsize=8)
plt.colorbar(im30,
            ax=ax30,
            fraction=0.0455, pad=0.04)

#4/4 area
ax31 = plt.subplot2grid((2,4), (1,3),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = dark.data,
            position = (dark.data.shape[1] - 500, dark.data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im31 = ax31.imshow(cut_hdu.data,
    origin='lower'
    )

ax31.set_ylabel('pixels')
ax31.grid(ls=':')
ax31.set_title(f'4/4 image area', fontsize=8)
plt.colorbar(im31,
            ax=ax31,
            fraction=0.0455, pad=0.04)

plt.tight_layout()
plt.show()

## Dealing with Flat

다음으로 생각해 보아야 할 것은 픽셀들 간의 양자효율에 대해서 생각해 보아야 합니다. 다시 말해 를 이루고 있는 픽셀들이 똑같은 빛에 대해서 똑같은 비율로 반응하는가 하는 점에 대해 생각해 볼 필요가 있습니다. 또한 광학계의 vignetting이나, filter 앞의 먼지 등으로 같은 밝기가 같은 픽셀값을 나타내지 않을 수 있습니다.

이 error는 다음과 같이 flat frame을 촬영하여 보정할 수 있습니다. flat frame을 촬영하여 bias와 dark를 제거하여 master flat을 만드는 과정은 다음과 같습니다.

* 해 질 무렵 동쪽 하늘이나 해 뜰 무렵 서쪽하을 이 flat 한 하늘로 볼 수 있습니다. $ADU$ 값이 25,000 전후가 되도록 flat image를 가능한 한 여려 장(8 ~ 10 frame) 촬영합니다.
* 필터를 사용할 경우 각각의 필터 마다 flat frame을 촬영해야 합니다.
* all flat image를 median 방법으로 combine "flat0-median.fits"로 저장합니다.
* flat0 값에서 bias 와 dark (dark should be scaled)
* save it as "flat.fits"
* save it as "flat-median.fits"

* 다만 저장시에 data type에 유의해야 하는데, ccdproc로 저장한후 data type을 확인해야 할 필요가 있습니다.

flat image에는 spatial variation이 있기 때문에 rejected pixel values 들을 모든 픽셀의 median value로 대체 할 수 없습니다. 보통은 lower limit of pixel value (`min_value`)를 설정하고 이 값보다 작은 픽셀 값을 바꾸는 것이 일반적인 방법입니다. 'min_value '보다 작은 픽셀을 'min_value'로 변경하겠습니다.

앞서 언급했지만 dark frame은 노출시간에 맞게 `dark(t)/t * t'` 의 방법으로 "scaled" 되어야 합니다. 이 과정은 `ccd_process` fucntion 의 "`dark_scale`" 에서 자동 조정됩니다.


### flat 파일 리스트 만들기

파일 이름에 규칙성이 있다면 이를 이용하여 다음과 같이 파일 경로를 리스트로 만들어서 사용할 수 있습니다.

In [ ]:
from glob import glob
import os

flat_list = sorted(list((BASEPATH/save_dir_name).glob('*flatV*.fit*')))
print(f"flat_list: {flat_list}")
print(f"len(flat_list): {len(flat_list)}")

In [ ]:
# from glob import glob
# import os

# flat_list = sorted(glob(os.path.join("./20170220_m35/", "flat*.fits")))
# print("flat_list : {}".format(flat_list))

### master flat0 만들기

ccdproc의 combime 함수를 이용하면 여러 장의 flat 이미지를 통계적으로 처리할 수 있습니다.

기본적으로는 여러장의 이미지로부터 각 픽셀의 통계값을 구하는데 기본은 average 값을 구하게 됩니다. 만약 median 값을 구하고자 하는 경우 method를 "median"을 입력해 주면 되며, 마스타 바이어스 프레임은 "median"이 더 유용할 것으로 생각됩니다.

In [ ]:
from ccdproc import CCDData, ccd_process, combine
import astropy.units as u

flat0 = combine(flat_list,       # ccdproc does not accept numpy.ndarray, but only python list.
               method='median',         # default is average so I specified median.
               unit='adu')

# float32로 변환
flat0.data = flat0.data.astype(np.float32)

### 저장하기

In [ ]:
# write fits file
flat0.write(f"{(BASEPATH/save_dir_name)}/master_flat0_{method}.fits", overwrite =True)
flat0.write(f"{(BASEPATH/save_dir_name)}/master_flat0_{method}.fits", overwrite =True)

### 화면에 출력하기 (flat0)

화면에 디스플레이 해 보자.

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(flat0, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(flat0.data,
            origin='lower',
            norm=norm,
                    )

im2 = axs[1].hist(flat0.data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'The master flat0 by median combine', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {flat0.data.min()}, Mean value: {flat0.data.mean():.02f}, Max value: {flat0.data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(flat0, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(flat0.data,
            origin='lower',
            #norm=norm,
            vmax = flat0.data.mean()*1.05,
                    )

im2 = axs[1].hist(flat0.data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'The master flat0 by median combine', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {flat0.data.min()}, Mean value: {flat0.data.mean():.02f}, Max value: {flat0.data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

### bias, dark 빼주기 (flat)

In [ ]:
# This dark isn't bias subtracted yet, so let's subtract bias:

# Subtract bias and dark
flat = ccd_process(flat0,                  # The input image (median combined flat)
                   master_bias = bias,       # Master bias
                   dark_frame = dark,        # dark
                   exposure_key='exptime', # the header keyword for exposure time
                   exposure_unit=u.s,      # the unit of exposure time
                   dark_scale=True)        # whether to scale dark frame

# float32로 변환
flat.data = flat.data.astype(np.float32)

#print('flat', flat.data.dtype, flat.data.min(), flat.data.max(), flat)

flat.write(f"{(BASEPATH/save_dir_name)}/master_flat_{method}.fits", overwrite =True)
flat.write(f"{(BASEPATH/save_dir_name)}/master_flat_{method}.fits", overwrite =True)

### 화면에 출력하기 (flat)

화면에 디스플레이 해 보자.

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(flat, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(flat.data,
            origin='lower',
            #vmax = 3000,
            norm=norm,
                    )

im2 = axs[1].hist(flat.data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'The master Flat by median combine', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {flat.data.min()}, Mean value: {flat.data.mean():.02f}, Max value: {flat.data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

### 부분 확대해서 확인하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from astropy.nddata import Cutout2D

# Create an ImageNormalize object
norm = simple_norm(flat.data, 'sqrt')

fig_set = plt.figure(figsize=(16, 8))

ax1 = plt.subplot2grid((2,4), (0,0),
                    colspan=2, rowspan=2,
                    fig=fig_set)
im1 = ax1.imshow(flat.data,
            origin='lower',
            norm=norm,
            )
plt.colorbar(im1,
             ax = ax1,
             fraction=0.0455, pad=0.04)

#1/4 area
ax20 = plt.subplot2grid((2,4), (0,2),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = flat.data,
            position = (500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im20 = ax20.imshow(cut_hdu.data,
    origin='lower'
    )

ax20.set_ylabel('pixels')
ax20.grid(ls=':')
ax20.set_title(f'1/4 image area', fontsize=8)
plt.colorbar(im20,
            ax=ax20,
            fraction=0.0455, pad=0.04)

#2/4 area
ax21 = plt.subplot2grid((2,4), (0,3),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = flat.data,
            position = (flat.data.shape[1] - 500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im21 = ax21.imshow(cut_hdu.data,
    origin='lower'
    )

ax21.set_ylabel('pixels')
ax21.grid(ls=':')
ax21.set_title(f'2/4 image area', fontsize=8)
plt.colorbar(im21,
            ax=ax21,
            fraction=0.0455, pad=0.04)

#3/4 area
ax30 = plt.subplot2grid((2,4), (1,2),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = flat.data,
            position = (500, flat.data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im30 = ax30.imshow(cut_hdu.data,
    origin='lower'
    )

ax30.set_ylabel('pixels')
ax30.grid(ls=':')
ax30.set_title(f'3/4 image area', fontsize=8)
plt.colorbar(im30,
            ax=ax30,
            fraction=0.0455, pad=0.04)

#4/4 area
ax31 = plt.subplot2grid((2,4), (1,3),
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = flat.data,
            position = (flat.data.shape[1] - 500, flat.data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
im31 = ax31.imshow(cut_hdu.data,
    origin='lower'
    )

ax31.set_ylabel('pixels')
ax31.grid(ls=':')
ax31.set_title(f'4/4 image area', fontsize=8)
plt.colorbar(im31,
            ax=ax31,
            fraction=0.0455, pad=0.04)

plt.tight_layout()
plt.show()

## Reducing Object Images


### 파일 리스트 만들기

파일 이름에 규칙성이 있다면 이를 이용하여 다음과 같이 파일 경로를 리스트로 만들어서 사용할 수 있습니다.

In [ ]:
from glob import glob
import os

object_list = sorted(list((BASEPATH/save_dir_name).glob('M35*.fit*')))
print(f"object_list: {object_list}")
print(f"len(object_list): {len(object_list)}")

In [ ]:
# from glob import glob
# import os

# object_list = sorted(glob(os.path.join("./20170220_m35/", "M35*.fits")))
# print("object_list : {}".format(object_list))

### 전처리 하기

위의 과정으로 master bias, master dark, 그리고 master flat (median combined flat - (bias + dark))을 만들었다면 간단하게 (object - (bias + dark)) / (master flat) 과정으로 calibration을 할 수 있습니다.

이 모든 preprocessing 과정을 한번에 하는 방법은 다음과 같다.

In [ ]:
import numpy as np
from ccdproc import CCDData, ccd_process
import astropy.units as u

# (1) Open master bias, dark, and flat
#bias = CCDData.read(f"{(BASEPATH/save_dir_name)}/master_bais_{method}.fits", unit='adu')
#dark = CCDData.read(f"{(BASEPATH/save_dir_name)}/master_dark_{method}.fits", unit='adu')
#flat = CCDData.read(f"{(BASEPATH/save_dir_name)}/master_flat_{method}.fits", unit='adu')  # Bias NOT subtracted
#flat0 = CCDData.read(f"{(BASEPATH/save_dir_name)}/master_flat0_{method}.fits", unit='adu')  # Bias NOT subtracted

# (2) Reduce each object image separately.
#     Then save it with prefix 'reduced_' which means "preprocessed"
for fpath in object_list:
    obj = CCDData.read(str(fpath), unit='adu')
    reduced = ccd_process(obj,                    # The input image
                          master_bias = bias,       # Master bias
                          dark_frame = dark,        # dark
                          master_flat = flat0,      # non-calibrated flat
                          min_value = flat.data.min(),        # flat.min() should be 30000
                          exposure_key = 'exptime', # exposure time keyword
                          exposure_unit = u.s,      # exposure time unit
                          dark_scale = True)        # whether to scale dark frame
    reduced.data = np.array(reduced.data, dtype=np.uint32)
    print('reduced', reduced.data.dtype, reduced.data.min(), reduced.data.max(), reduced)
    reduced.write(f"{(BASEPATH/save_dir_name)}/reduced_{fpath.name}", overwrite =True)
    print(f"{(BASEPATH/save_dir_name)}/reduced_{fpath.name} is created...")

### 전처리 결과

전처리한 결과를 비교해 보겠습니다.


In [ ]:
# read file
fpath = object_list[0]
print(fpath)
hdul = fits.open(str(fpath))
print("hdul : {}".format(hdul))

import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(hdul[0].data, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(hdul[0].data,
            origin='lower',
            #norm=norm,
            vmax = hdul[0].data.mean()*1.05,
                    )

im2 = axs[1].hist(hdul[0].data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'Before preprocessing', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {hdul[0].data.min()}, Mean value: {hdul[0].data.mean():.02f}, Max value: {hdul[0].data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()


In [ ]:
# read file
fpath = object_list[0]
reduced_fpath = (BASEPATH/save_dir_name)/f"reduced_{fpath.name}"
print(reduced_fpath)
hdul = fits.open(str(reduced_fpath))

import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

# Create an ImageNormalize object
norm = simple_norm(hdul[0].data, 'sqrt')

fig, axs = plt.subplots(1, 2, figsize=(13, 6),
                        sharex=False, sharey=False, gridspec_kw=None)

im1 = axs[0].imshow(hdul[0].data,
            origin='lower',
            #norm=norm,
            vmax = hdul[0].data.mean()*1.05,
                    )

im2 = axs[1].hist(hdul[0].data,
                    # 50,
                    # histtype='step',
                    )

axs[0].set_title(f'After preprocessing', fontsize=9)
axs[1].set_title('The histrogram of pixel value', fontsize=9)

plt.colorbar(im1, fraction=0.0455, pad=0.04)
plt.annotate(f"Min value: {hdul[0].data.min()}, Mean value: {hdul[0].data.mean():.02f}, Max value: {hdul[0].data.max()}",
             xy=(0, -100), xycoords='axes pixels')
plt.tight_layout(pad=1.0)
plt.show()

### 부분 확대해서 확인하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import pylab as pl
import matplotlib.patches as patches
from astropy.visualization import simple_norm
from astropy.nddata import Cutout2D

# read file
hdul = fits.open(str(fpath))
print("hdul : {}".format(hdul))

# Create an ImageNormalize object
norm = simple_norm(hdul[0].data, 'sqrt')

fig_set = plt.figure(figsize=(16, 8))

ax1 = plt.subplot2grid((2,4), (0,0),
                    colspan=2, rowspan=2,
                    fig=fig_set)
im1 = ax1.imshow(hdul[0].data,
            origin='lower',
            #norm=norm,
            vmax = hdul[0].data.mean()*1.05,
            )
ax1.set_title(f'Before preprocessing', fontsize=12)
plt.colorbar(im1,
             ax = ax1,
             fraction=0.0455, pad=0.04)

#1/4 area
ax20 = plt.subplot2grid((2,4), (0,2),
                        projection='3d',
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = hdul[0].data,
            position = (500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
x, y = np.mgrid[:cut_hdu.data.shape[0],:cut_hdu.data.shape[1]]
im20 = ax20.plot_surface(x, y, cut_hdu.data, cmap=pl.cm.jet,
                         rstride=1,cstride=1,linewidth=0.,antialiased=False)

ax20.set_xlabel('pixels')
ax20.set_ylabel(f'mean value: {cut_hdu.data.mean():.02f}', fontsize=8)
ax20.grid(ls=':')
ax20.set_title(f'1/4 image area', fontsize=8)
plt.colorbar(im20,
            ax=ax20,
            fraction=0.025, pad=0.12)

# draw retangle
rect = patches.Rectangle((500, 500), cutsizes, cutsizes, linewidth=1, edgecolor='r', facecolor='none')
# Add the patch to the Axes
ax1.add_patch(rect)
ax1.text(500, 500, "1/4")


#2/4 area
ax21 = plt.subplot2grid((2,4), (0,3),
                        projection='3d',
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = hdul[0].data,
            position = (hdul[0].data.shape[1] - 500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
x, y = np.mgrid[:cut_hdu.data.shape[0],:cut_hdu.data.shape[1]]
im21 = ax21.plot_surface(x, y, cut_hdu.data, cmap=pl.cm.jet,
                         rstride=1,cstride=1,linewidth=0.,antialiased=False)

ax21.set_xlabel('pixels')
ax21.set_ylabel(f'mean value: {cut_hdu.data.mean():.02f}', fontsize=8)
ax21.grid(ls=':')
ax21.set_title(f'2/4 image area', fontsize=8)
plt.colorbar(im21,
            ax=ax21,
            fraction=0.025, pad=0.12)
# draw retangle
rect = patches.Rectangle((hdul[0].data.shape[1] - 500, 500), cutsizes, cutsizes, linewidth=1, edgecolor='r', facecolor='none')
# Add the patch to the Axes
ax1.add_patch(rect)
ax1.text(hdul[0].data.shape[1] - 500, 500, "2/4")

#3/4 area
ax30 = plt.subplot2grid((2,4), (1,2),
                        projection='3d',
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = hdul[0].data,
            position = (500, hdul[0].data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
x, y = np.mgrid[:cut_hdu.data.shape[0],:cut_hdu.data.shape[1]]
im30 = ax30.plot_surface(x, y, cut_hdu.data, cmap=pl.cm.jet,
                         rstride=1,cstride=1,linewidth=0.,antialiased=False)

ax30.set_xlabel('pixels')
ax30.set_ylabel(f'mean value: {cut_hdu.data.mean():.02f}', fontsize=8)
ax30.grid(ls=':')
ax30.set_title(f'3/4 image area', fontsize=8)
plt.colorbar(im30,
            ax=ax30,
            fraction=0.025, pad=0.12)
# draw retangle
rect = patches.Rectangle((500, hdul[0].data.shape[0] - 500), cutsizes, cutsizes, linewidth=1, edgecolor='r', facecolor='none')
# Add the patch to the Axes
ax1.add_patch(rect)
ax1.text(500, hdul[0].data.shape[0] - 500, "3/4")

#4/4 area
ax31 = plt.subplot2grid((2,4), (1,3),
                        projection='3d',
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = hdul[0].data,
            position = (hdul[0].data.shape[1] - 500, hdul[0].data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
x, y = np.mgrid[:cut_hdu.data.shape[0],:cut_hdu.data.shape[1]]
im31 = ax31.plot_surface(x, y, cut_hdu.data, cmap=pl.cm.jet,
                         rstride=1,cstride=1,linewidth=0.,antialiased=False)

ax31.set_xlabel('pixels')
ax31.set_ylabel(f'mean value: {cut_hdu.data.mean():.02f}', fontsize=8)
ax31.grid(ls=':')
ax31.set_title(f'4/4 image area', fontsize=8)
plt.colorbar(im31,
            ax=ax31,
            fraction=0.025, pad=0.12)
# draw retangle
rect = patches.Rectangle((hdul[0].data.shape[1] - 500, hdul[0].data.shape[0] - 500), cutsizes, cutsizes, linewidth=1, edgecolor='r', facecolor='none')
# Add the patch to the Axes
ax1.add_patch(rect)
ax1.text(hdul[0].data.shape[1] - 500, hdul[0].data.shape[0] - 500, "4/4")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import pylab as pl
import matplotlib.patches as patches
from astropy.visualization import simple_norm
from astropy.nddata import Cutout2D

# read file
hdul = fits.open(str(reduced_fpath))
print("hdul : {}".format(hdul))

# Create an ImageNormalize object
norm = simple_norm(hdul[0].data, 'sqrt')

fig_set = plt.figure(figsize=(16, 8))

ax1 = plt.subplot2grid((2,4), (0,0),
                    colspan=2, rowspan=2,
                    fig=fig_set)
im1 = ax1.imshow(hdul[0].data,
            origin='lower',
            #norm=norm,
            vmax = hdul[0].data.mean()*1.05,
            )
ax1.set_title(f'Before preprocessing', fontsize=12)
plt.colorbar(im1,
             ax = ax1,
             fraction=0.0455, pad=0.04)

#1/4 area
ax20 = plt.subplot2grid((2,4), (0,2),
                        projection='3d',
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = hdul[0].data,
            position = (500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
x, y = np.mgrid[:cut_hdu.data.shape[0],:cut_hdu.data.shape[1]]
im20 = ax20.plot_surface(x, y, cut_hdu.data, cmap=pl.cm.jet,
                         rstride=1,cstride=1,linewidth=0.,antialiased=False)

ax20.set_xlabel('pixels')
ax20.set_ylabel(f'mean value: {cut_hdu.data.mean():.02f}', fontsize=8)
ax20.grid(ls=':')
ax20.set_title(f'1/4 image area', fontsize=8)
plt.colorbar(im20,
            ax=ax20,
            fraction=0.025, pad=0.12)

# draw retangle
rect = patches.Rectangle((500, 500), cutsizes, cutsizes, linewidth=1, edgecolor='r', facecolor='none')
# Add the patch to the Axes
ax1.add_patch(rect)
ax1.text(500, 500, "1/4")


#2/4 area
ax21 = plt.subplot2grid((2,4), (0,3),
                        projection='3d',
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = hdul[0].data,
            position = (hdul[0].data.shape[1] - 500, 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
x, y = np.mgrid[:cut_hdu.data.shape[0],:cut_hdu.data.shape[1]]
im21 = ax21.plot_surface(x, y, cut_hdu.data, cmap=pl.cm.jet,
                         rstride=1,cstride=1,linewidth=0.,antialiased=False)

ax21.set_xlabel('pixels')
ax21.set_ylabel(f'mean value: {cut_hdu.data.mean():.02f}', fontsize=8)
ax21.grid(ls=':')
ax21.set_title(f'2/4 image area', fontsize=8)
plt.colorbar(im21,
            ax=ax21,
            fraction=0.025, pad=0.12)
# draw retangle
rect = patches.Rectangle((hdul[0].data.shape[1] - 500, 500), cutsizes, cutsizes, linewidth=1, edgecolor='r', facecolor='none')
# Add the patch to the Axes
ax1.add_patch(rect)
ax1.text(hdul[0].data.shape[1] - 500, 500, "2/4")

#3/4 area
ax30 = plt.subplot2grid((2,4), (1,2),
                        projection='3d',
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = hdul[0].data,
            position = (500, hdul[0].data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
x, y = np.mgrid[:cut_hdu.data.shape[0],:cut_hdu.data.shape[1]]
im30 = ax30.plot_surface(x, y, cut_hdu.data, cmap=pl.cm.jet,
                         rstride=1,cstride=1,linewidth=0.,antialiased=False)

ax30.set_xlabel('pixels')
ax30.set_ylabel(f'mean value: {cut_hdu.data.mean():.02f}', fontsize=8)
ax30.grid(ls=':')
ax30.set_title(f'3/4 image area', fontsize=8)
plt.colorbar(im30,
            ax=ax30,
            fraction=0.025, pad=0.12)
# draw retangle
rect = patches.Rectangle((500, hdul[0].data.shape[0] - 500), cutsizes, cutsizes, linewidth=1, edgecolor='r', facecolor='none')
# Add the patch to the Axes
ax1.add_patch(rect)
ax1.text(500, hdul[0].data.shape[0] - 500, "3/4")

#4/4 area
ax31 = plt.subplot2grid((2,4), (1,3),
                        projection='3d',
                       fig=fig_set)

cutsizes = 101
cut_hdu = Cutout2D(
            data = hdul[0].data,
            position = (hdul[0].data.shape[1] - 500, hdul[0].data.shape[0] - 500),
            size=(cutsizes, cutsizes) #cut ccd
            )
x, y = np.mgrid[:cut_hdu.data.shape[0],:cut_hdu.data.shape[1]]
im31 = ax31.plot_surface(x, y, cut_hdu.data, cmap=pl.cm.jet,
                         rstride=1,cstride=1,linewidth=0.,antialiased=False)

ax31.set_xlabel('pixels')
ax31.set_ylabel(f'mean value: {cut_hdu.data.mean():.02f}', fontsize=8)
ax31.grid(ls=':')
ax31.set_title(f'4/4 image area', fontsize=8)
plt.colorbar(im31,
            ax=ax31,
            fraction=0.025, pad=0.12)
# draw retangle
rect = patches.Rectangle((hdul[0].data.shape[1] - 500, hdul[0].data.shape[0] - 500), cutsizes, cutsizes, linewidth=1, edgecolor='r', facecolor='none')
# Add the patch to the Axes
ax1.add_patch(rect)
ax1.text(hdul[0].data.shape[1] - 500, hdul[0].data.shape[0] - 500, "4/4")

plt.tight_layout()
plt.show()

전처리가 완료된 영상을 볼 수 있습니다.

##(과졔)

선생님이 배부해 주는 파일을 받아서 전처리 완료후에 제출해 보자.
전처리된 영상은 별도의 파일로 하나로 압축하여 첨부해 주세요.